<a href="https://colab.research.google.com/github/UtkarshSharma-004/X-ray-lungs-disease-Detection/blob/main/Copy_of_CapstoneProject(CNN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
path = kagglehub.dataset_download("nih-chest-xrays/data")

100%|██████████| 42.0G/42.0G [08:07<00:00, 92.6MB/s]

Extracting files...


In [ ]:
path

In [ ]:
import shutil

print("Downloaded to:", path)

# Destination folder in Colab
destination = "/content/drive/MyDrive/Capstone Project/nih_chest_xrays"

# Move dataset to /content
shutil.copytree(path, destination, dirs_exist_ok=True)

print("Dataset copied to:", destination)

Downloaded to: /root/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3


In [ ]:
destination = "/content/drive/MyDrive/Capstone Project/nih_chest_xrays"

In [ ]:
# # Delete the original downloaded folder
# shutil.rmtree(path)

# print("Original dataset deleted from:", path)

Original dataset deleted from: /root/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3


In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df= pd.read_csv(f'{destination}/Data_Entry_2017.csv')

In [ ]:
df.sample(10)

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
86279,00021277_003.png,Nodule|Pleural_Thickening,3,21277,19,M,AP,2956,2544,0.139,0.139,NaN
32979,00008636_000.png,Mass,0,8636,58,F,AP,2500,2048,0.171,0.171,NaN
51082,00012907_010.png,No Finding,10,12907,40,M,PA,2992,2991,0.143,0.143,NaN
9961,00002582_010.png,Atelectasis|Effusion,10,2582,62,M,AP,2500,2048,0.168,0.168,NaN
104578,00027982_005.png,No Finding,5,27982,36,M,AP,3056,2544,0.139,0.139,NaN
64648,00015956_021.png,Infiltration,21,15956,67,M,AP,2500,2048,0.168,0.168,NaN
102801,00027415_034.png,Infiltration,34,27415,23,M,AP,3056,2544,0.139,0.139,NaN
99486,00026332_000.png,Atelectasis|Effusion,0,26332,55,M,PA,3056,2544,0.139,0.139,NaN
88110,00021796_007.png,Effusion,7,21796,54,M,AP,3056,2544,0.139,0.139,NaN
56607,00014083_000.png,No Finding,0,14083,22,F,PA,2990,2991,0.143,0.143,NaN


In [ ]:
df['Finding Labels'].value_counts()

,count
Finding Labels,
No Finding,60361
Infiltration,9547
Atelectasis,4215
Effusion,3955
Nodule,2705
...,...
Hernia|Infiltration|Nodule,1
Cardiomegaly|Hernia|Mass,1
Atelectasis|Effusion|Infiltration|Pneumonia|Pneumothorax,1


In [ ]:
df[df['Finding Labels']=='No Finding']

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
13,00000005_000.png,No Finding,0,5,69,F,PA,2048,2500,0.168,0.168,NaN
14,00000005_001.png,No Finding,1,5,69,F,AP,2500,2048,0.168,0.168,NaN
15,00000005_002.png,No Finding,2,5,69,F,AP,2500,2048,0.168,0.168,NaN
16,00000005_003.png,No Finding,3,5,69,F,PA,2992,2991,0.143,0.143,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
112114,00030801_000.png,No Finding,0,30801,39,M,PA,2500,2048,0.168,0.168,NaN
112116,00030802_000.png,No Finding,0,30802,29,M,PA,2048,2500,0.168,0.168,NaN
112117,00030803_000.png,No Finding,0,30803,42,F,PA,2048,2500,0.168,0.168,NaN
112118,00030804_000.png,No Finding,0,30804,30,F,PA,2048,2500,0.168,0.168,NaN


In [ ]:
import os

imageFolderPaths = []


for folderName in os.listdir(destination):
  # print(folderName)
  if folderName.startswith("images_"):
    folderPath = os.path.join(destination, folderName, "images")
    if os.path.isdir(folderPath):
        imageFolderPaths.append(folderPath)

imageFolderPaths = sorted(imageFolderPaths)
print(imageFolderPaths)

['/content/drive/MyDrive/Capstone Project/nih_chest_xrays/images_001/images', '/content/drive/MyDrive/Capstone Project/nih_chest_xrays/images_002/images', '/content/drive/MyDrive/Capstone Project/nih_chest_xrays/images_003/images', '/content/drive/MyDrive/Capstone Project/nih_chest_xrays/images_005/images', '/content/drive/MyDrive/Capstone Project/nih_chest_xrays/images_006/images', '/content/drive/MyDrive/Capstone Project/nih_chest_xrays/images_007/images', '/content/drive/MyDrive/Capstone Project/nih_chest_xrays/images_008/images', '/content/drive/MyDrive/Capstone Project/nih_chest_xrays/images_009/images', '/content/drive/MyDrive/Capstone Project/nih_chest_xrays/images_010/images', '/content/drive/MyDrive/Capstone Project/nih_chest_xrays/images_011/images', '/content/drive/MyDrive/Capstone Project/nih_chest_xrays/images_012/images']


In [ ]:
len(imageFolderPaths)

12

In [ ]:
imageRows = []

for folderPath in imageFolderPaths:
    for imageName in os.listdir(folderPath):
        imageRows.append({
            "imageIndex": imageName,
            "imagePath": os.path.join(folderPath, imageName)
        })

imagePathDf = pd.DataFrame(imageRows)

print(imagePathDf.shape)
imagePathDf.head()

(101038, 2)


,imageIndex,imagePath
0,00000782_002.png,/content/drive/MyDrive/Capstone Project/nih_ch...
1,00001314_000.png,/content/drive/MyDrive/Capstone Project/nih_ch...
2,00000761_004.png,/content/drive/MyDrive/Capstone Project/nih_ch...
3,00000798_024.png,/content/drive/MyDrive/Capstone Project/nih_ch...
4,00000859_000.png,/content/drive/MyDrive/Capstone Project/nih_ch...


In [ ]:
df.rename(columns={'Image Index':'imageIndex'},inplace=True)

In [ ]:
df.columns

Index(['imageIndex', 'Finding Labels', 'Follow-up #', 'Patient ID',
       'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width',
       'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11'],
      dtype='object')

In [ ]:
masterDf = df.merge(imagePathDf, on="imageIndex", how="left")
masterDf = masterDf.dropna(subset=["imagePath"]).reset_index(drop=True)

print(masterDf.shape)
print("Missing imagePath:", masterDf["imagePath"].isna().sum())
masterDf.head()

(101038, 13)
Missing imagePath: 0


,imageIndex,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11,imagePath
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...


In [ ]:
masterDf[masterDf['Finding Labels'].isin(['No Finding','Infiltration'])].columns

Index(['imageIndex', 'Finding Labels', 'Follow-up #', 'Patient ID',
       'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width',
       'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11',
       'imagePath'],
      dtype='object')

In [ ]:
filteredDf = masterDf[masterDf['Finding Labels'].isin(['No Finding','Infiltration'])].copy()

print(filteredDf.shape)
filteredDf.head()

(62888, 13)


,imageIndex,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11,imagePath
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...
13,00000005_000.png,No Finding,0,5,69,F,PA,2048,2500,0.168,0.168,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...
14,00000005_001.png,No Finding,1,5,69,F,AP,2500,2048,0.168,0.168,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...
15,00000005_002.png,No Finding,2,5,69,F,AP,2500,2048,0.168,0.168,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...
16,00000005_003.png,No Finding,3,5,69,F,PA,2992,2991,0.143,0.143,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...


In [ ]:
filteredDf['label'] = filteredDf['Finding Labels'].map({
    'No Finding':0,
    'Infiltration':1
})

In [ ]:
filteredDf

,imageIndex,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11,imagePath,label
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...,0
13,00000005_000.png,No Finding,0,5,69,F,PA,2048,2500,0.168,0.168,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...,0
14,00000005_001.png,No Finding,1,5,69,F,AP,2500,2048,0.168,0.168,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...,0
15,00000005_002.png,No Finding,2,5,69,F,AP,2500,2048,0.168,0.168,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...,0
16,00000005_003.png,No Finding,3,5,69,F,PA,2992,2991,0.143,0.143,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101032,00030801_000.png,No Finding,0,30801,39,M,PA,2500,2048,0.168,0.168,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...,0
101034,00030802_000.png,No Finding,0,30802,29,M,PA,2048,2500,0.168,0.168,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...,0
101035,00030803_000.png,No Finding,0,30803,42,F,PA,2048,2500,0.168,0.168,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...,0
101036,00030804_000.png,No Finding,0,30804,30,F,PA,2048,2500,0.168,0.168,NaN,/content/drive/MyDrive/Capstone Project/nih_ch...,0


In [ ]:
import os

dataset_dir = "/content/drive/MyDrive/Capstone Project/chest_dataset"
os.makedirs(dataset_dir, exist_ok=True)

os.makedirs(dataset_dir + "/No_Finding", exist_ok=True)
os.makedirs(dataset_dir + "/Infiltration", exist_ok=True)

In [ ]:
filteredDf.to_csv("/content/drive/MyDrive/Capstone Project/chest_dataset_labels.csv", index=False)

In [ ]:
import shutil
import os

for _, row in filteredDf.iterrows():

    img_path = row['imagePath']
    label = row['Finding Labels']

    # choose destination folder
    if label == "No Finding":
        dest = os.path.join(dataset_dir, "No_Finding")
    else:
        dest = os.path.join(dataset_dir, "Infiltration")

    # get filename
    filename = os.path.basename(img_path)

    dest_path = os.path.join(dest, filename)

    # check if file already exists
    if not os.path.exists(dest_path):
        shutil.copy(img_path, dest_path)
    else:
        print(f"Skipped (already exists): {filename}")

Streaming output truncated to the last 5000 lines.
Skipped (already exists): 00027752_000.png
Skipped (already exists): 00027753_000.png
Skipped (already exists): 00027754_000.png
Skipped (already exists): 00027755_000.png
Skipped (already exists): 00027756_000.png
Skipped (already exists): 00027757_000.png
Skipped (already exists): 00027757_001.png
Skipped (already exists): 00027759_000.png
Skipped (already exists): 00027762_000.png
Skipped (already exists): 00027763_000.png
Skipped (already exists): 00027763_001.png
Skipped (already exists): 00027764_000.png
Skipped (already exists): 00027765_000.png
Skipped (already exists): 00027765_001.png
Skipped (already exists): 00027765_002.png
Skipped (already exists): 00027766_000.png
Skipped (already exists): 00027767_000.png
Skipped (already exists): 00027768_000.png
Skipped (already exists): 00027769_000.png
Skipped (already exists): 00027770_000.png
Skipped (already exists): 00027771_000.png
Skipped (already exists): 00027773_000.png
Ski